# Tâche 10: Splits train/validation/test

**Projet 1 · African Language Data & LLM Readiness · AfrikLang**

Ajoute un champ `split` (`train` / `validation` / `test`) à `data/processed/provenance/<langue>.jsonl` et
`provenance.parquet` le corpus (texte) n'est jamais modifié.

- Répartition **par langue** : 80 % train / 10 % validation / 10 % test.
- Langues en dessous de 20 textes : 100 % train (val/test vides), justifié même logique que les langues à
  0 texte de la tâche 9.
- Anti-fuite : les quasi-doublons (tâche 7.2) et les doublons exacts entre langues (tâche 7.1) jamais
  retirés du corpus, seulement signalés sont regroupés en clusters qui reçoivent toujours un seul et même
  split, pour qu'aucun texte et son quasi-doublon ne se retrouvent de part et d'autre de la frontière
  train/test.
- Reproductible : attribution par hachage de l'empreinte du texte (ou du groupe) + `random_seed`, sans état
  aléatoire à conserver.

In [ ]:
%pip install -q -r ../requirements.txt

In [1]:
import json, os, subprocess, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ANALYSES = ROOT / "data" / "processed" / "qualite" / "analyses"


def lancer(*args):
    # Lance src/split.py et affiche son journal au fur et à mesure.
    env = dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONUTF8="1")
    p = subprocess.Popen([sys.executable, "-u", str(ROOT / "src" / "split.py"), *args], cwd=ROOT, env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    try:
        for ligne in p.stdout:
            print(ligne, end="")
    except KeyboardInterrupt:
        p.terminate()
        print("\nArrêté.")
    return p.wait()


def derniere(nom):
    # Dernière analyse d'un type donné (ici : split)
    return sorted(ANALYSES.glob(f"*_{nom}"))[-1]


## Lancer le split

In [7]:
lancer()
d = derniere("split")
b = json.loads((d / "bilan.json").read_text(encoding="utf-8"))
print(f"{b['textes']} textes -> {b['train']} train, {b['validation']} validation, {b['test']} test")
print(f"{b['groupes_anti_fuite']} groupe(s) anti-fuite ({b['textes_regroupes']} textes concernés)")
print("langues sous le seuil (tout en train) :", b["langues_sous_seuil_tout_en_train"] or "aucune")
print("anomalies :", b["anomalies"] or "aucune")
tab = pd.DataFrame(b["langues"]).set_index("langue")
tab


2026-11-23T22:16:15 INFO 10.1 Reconstitution des groupes anti-fuite (quasi-doublons tâche 7.2, doublons inter-langues tâche 7.1) pour 69 langue(s)
2026-11-23T22:16:36 WARNING Membres de groupe non retrouvés dans le corpus final (déjà écartés à l'export, tâche 8) : {'doublons_langues': 2}
2026-11-23T22:16:36 INFO 10.2 Attribution des splits : ratios {'train': 0.8, 'validation': 0.1, 'test': 0.1}, seuil langue min 20 textes, seed 42
2026-11-23T22:16:36 INFO 10.3 Écriture du champ split, une langue à la fois (la plus grosse d'abord), remplacement immédiat de l'ancien fichier — jamais plus d'une langue dupliquée sur le disque à la fois
2026-11-23T22:20:00 INFO arz   1631258 textes : 1304797 train, 163618 validation, 162843 test  (204 s)
2026-11-23T22:22:47 INFO ar    1332977 textes : 1065826 train, 133381 validation, 133770 test  (168 s)
2026-11-23T22:23:04 INFO af     131091 textes :  104751 train,  13253 validation,  13087 test  (16 s)
2026-11-23T22:23:20 INFO sw     129081 textes :  103

,textes,force_train,train,validation,test,duree_s
langue,,,,,,
aa,0,True,0,0,0,0
af,131091,False,104751,13253,13087,16
ak,1,True,1,0,0,0
am,16286,False,12983,1636,1667,2
ann,494,False,388,60,46,0
...,...,...,...,...,...,...
wo,1809,False,1419,196,194,0
xh,3090,False,2457,326,307,0
yo,41803,False,33463,4194,4146,5


## Vérification anti-fuite

`bilan.json` confirme déjà (champ `anomalies`) qu'aucun groupe de quasi-doublons ni de doublons inter-langues
ne se retrouve à cheval entre deux splits, et qu'aucune langue sous le seuil n'a de texte hors `train`. La
cellule suivante relit `clusters_regroupes.jsonl` pour un aperçu de ces groupes et du split qu'ils ont reçu.

In [3]:
clusters = pd.read_json(d / "clusters_regroupes.jsonl", lines=True)
print(f"{len(clusters)} groupe(s) anti-fuite (quasi-doublons + doublons inter-langues)")
clusters.head(10)


3944 groupe(s) anti-fuite (quasi-doublons + doublons inter-langues)


,membres,split
0,"[{'langue': 'af', 'index': 4445}, {'langue': '...",train
1,"[{'langue': 'af', 'index': 40811}, {'langue': ...",train
2,"[{'langue': 'af', 'index': 40819}, {'langue': ...",train
3,"[{'langue': 'af', 'index': 64849}, {'langue': ...",train
4,"[{'langue': 'af', 'index': 82847}, {'langue': ...",train
5,"[{'langue': 'af', 'index': 83621}, {'langue': ...",train
6,"[{'langue': 'af', 'index': 88733}, {'langue': ...",train
7,"[{'langue': 'am', 'index': 397}, {'langue': 'a...",train
8,"[{'langue': 'am', 'index': 3540}, {'langue': '...",test
9,"[{'langue': 'am', 'index': 3546}, {'langue': '...",train


## Vue d'ensemble

In [4]:
# Part de chaque split par langue (langues avec le plus de textes)
vue = tab.copy()
for s in ("train", "validation", "test"):
    vue[f"part_{s}"] = (vue[s] / vue["textes"]).round(3)
vue.sort_values("textes", ascending=False)[["textes", "train", "validation", "test",
                                            "part_train", "part_validation", "part_test"]].head(15)


,textes,train,validation,test,part_train,part_validation,part_test
langue,,,,,,,
arz,1631258,1304797,163618,162843,0.800,0.100,0.100
ar,1332977,1065826,133381,133770,0.800,0.100,0.100
af,131091,104751,13253,13087,0.799,0.101,0.100
sw,129081,103330,12928,12823,0.801,0.100,0.099
ha,123650,98780,12421,12449,0.799,0.100,0.101
mg,104097,83330,10448,10319,0.801,0.100,0.099
fr,99961,79826,10030,10105,0.799,0.100,0.101
ig,56485,45121,5749,5615,0.799,0.102,0.099
yo,41803,33463,4194,4146,0.800,0.100,0.099


In [5]:
# Aperçu du registre de provenance final (Parquet), avec le champ split
pd.read_parquet(ROOT / "data" / "processed" / "provenance.parquet").sample(10, random_state=42)


,text_sha256,language_code,source_url,source_dump,revision_date,collected_at,license,quality_flags,split
3462519,0a6b9a616c6b168e99a61f1b520a38d97cd4b7089d4b1e...,ig,https://ig.wikipedia.org/w/index.php?oldid=177945,igwiki-20261001-pages-articles,2024-07-07T07:33:57Z,2026-11-05T20:52:06Z,CC-BY-SA-4.0,[],train
3530918,2902924c1f4828b4771a2a28729e20673131064d186497...,mg,https://mg.wikipedia.org/w/index.php?oldid=921426,mgwiki-20261001-pages-articles,2018-10-29T11:25:00Z,2026-11-05T20:43:35Z,CC-BY-SA-4.0,[template_generated],validation
2183652,1b4465665f356af4b3e47a0a689f615f5e027d3fc4e843...,arz,https://arz.wikipedia.org/w/index.php?oldid=12...,arzwiki-20261001-pages-articles,2025-05-31T11:32:59Z,2026-11-05T21:09:26Z,CC-BY-SA-4.0,"[boilerplate, template_generated]",train
3269536,4bea6406372ae5829cfcc5b10fb54c03d4371966d5430c...,fr,https://fr.wikipedia.org/w/index.php?oldid=227...,api-20261105,2025-08-08T12:13:31Z,2026-11-05T23:11:58Z,CC-BY-SA-4.0,[],train
1322328,3583def481cfcb276b4bde1a3becc5ebcecb9313d07695...,ar,https://ar.wikipedia.org/w/index.php?oldid=711...,arwiki-20261001-pages-articles,2025-06-16T07:45:30Z,2026-11-05T22:12:31Z,CC-BY-SA-4.0,[language_mismatch],train
2203942,76b00a41de43a9a10e8ae5485e9405519a50229d5bc129...,arz,https://arz.wikipedia.org/w/index.php?oldid=11...,arzwiki-20261001-pages-articles,2025-04-15T18:18:52Z,2026-11-05T21:09:39Z,CC-BY-SA-4.0,[],test
2825693,4fc821410335e8faa314a5198a41dea58b1f67f82733f4...,arz,https://arz.wikipedia.org/w/index.php?oldid=11...,arzwiki-20261001-pages-articles,2025-04-30T11:00:37Z,2026-11-05T21:15:02Z,CC-BY-SA-4.0,[],train
2889570,259578ea322226866be381e24f7ce47dacee856d8fbf45...,arz,https://arz.wikipedia.org/w/index.php?oldid=11...,arzwiki-20261001-pages-articles,2025-04-14T18:09:22Z,2026-11-05T21:15:45Z,CC-BY-SA-4.0,[],train
625160,3619fe84f05c2947ee1903a4e69264357bef565297c9c6...,ar,https://ar.wikipedia.org/w/index.php?oldid=740...,arwiki-20261001-pages-articles,2026-03-19T01:44:38Z,2026-11-05T21:51:31Z,CC-BY-SA-4.0,[language_mismatch],train
3060140,6af32cac22279adf5f26272980b215fad200d09ea1075e...,arz,https://arz.wikipedia.org/w/index.php?oldid=97...,arzwiki-20261001-pages-articles,2024-03-30T10:50:42Z,2026-11-05T21:16:33Z,CC-BY-SA-4.0,[],train
